In [ ]:

# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0" # TODO = ?
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_DECODING"] = "0"
os.environ["OPENCV_VIDEOIO_MSMF_FORCE_BGR"] = "1"

# recording
from lerobot.record import RecordConfig, DatasetRecordConfig, record

# robot configs
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

# my code
from src.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME

# set up env secrets
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env", override=True)

True

Recording Parameters:

In [10]:
NUM_EPISODES = 5
FPS = 30
EPISODE_TIME_SEC = 60 # how long each episode
RESET_TIME_SEC   = 10 # env reset time
TASK_DESCRIPTION = 'test'
REPO_NAME = 'test'
PUSH_TO_HUB = False

Login to HF if pushing:

In [ ]:
if PUSH_TO_HUB:
    os.system(f"huggingface-cli login --token {os.getenv('HUGGINGFACE_TOKEN')}")

Robot Config:

In [ ]:
camera_config = {
    "wrist_cam": OpenCVCameraConfig(index_or_path=0, width=640, height=480, fps=30),
    "top_cam": OpenCVCameraConfig(index_or_path=1, width=640, height=480, fps=30),
}

robot_config = SO101FollowerConfig(
    port="COM7",
    id="so_101_follower",
    cameras = camera_config,
    calibration_dir = CALIBS_DIR
)

teleop_config = SO101LeaderConfig(
    port="COM8",
    id="so_101_leader",
    calibration_dir = CALIBS_DIR
)
robot = SO101Follower(robot_config)
teleop = SO101Leader(teleop_config)

Dataset Config:

In [13]:
dataset_path = DATASETS_DIR / REPO_NAME
repo_id=f"{HF_NAME}/{REPO_NAME}"
root=f"{DATASETS_DIR}\\{REPO_NAME}"
resume = True if dataset_path.exists() else False

In [ ]:
dscfg = DatasetRecordConfig(
    repo_id                             = repo_id,
    single_task                         = TASK_DESCRIPTION,
    root                                = root,
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 2,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_config,
    dataset      = dscfg,
    teleop       = teleop_config,
    policy       = None,
    display_data = True,
    resume       = resume
)

In [16]:
record(rc, save_to_ds = False, give_feedback = False, reset_pose = None)

WARNING 2025-09-23 21:30:37 deo_utils.py:40 'torchcodec' is not available in your platform, falling back to 'pyav' as a default decoder


ConnectionError: 
Could not connect on port 'COM7'. Make sure you are using the correct port.
Try running `lerobot-find-port`


In [ ]:
# robot.disconnect()
# teleop.disconnect()